# 28. Export and import a `DecayModel`

**Objectives:**
- Build a small multi-component `DecayModel` and serialize it with `export_model`.
- Reload it with `import_model` and check the reloaded model's `.intensity(...)`
  matches the original's, bit for bit, on a generated toy sample.
- Note the one unsupported case: `normalization_method="toy-mc"`.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero. See `docs/model_io.md` for the full specification.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance, export_model,
    generate_toy, import_model,
)

## 1. Build a small multi-component model

Two resonances on the same pair plus a non-resonant term -- enough to exercise
component, lineshape, angular and coefficient serialization without a long build.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
model = DecayModel(
    channel,
    [
        Resonance("rho", pair=(0, 1), coefficient=RealImag(1.0, 0.0),
                  mass=0.77526, width=0.1491, spin=1),
        Resonance("f2", pair=(0, 1), coefficient=RealImag(0.4, -0.2),
                  mass=1.2755, width=0.1867, spin=2),
        NonResonant(RealImag(0.3, 0.1), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=40,
)
print("Components:", [c.name for c in model.components])

Components: ['rho', 'f2', 'NR']


## 2. Export, reload, and compare intensities

`export_model` writes the `DecayChannel`, every component (lineshape/angular/
coefficient plugins and any embedded `Parameter`s), and the model-wide
`normalization_*` settings to JSON. `import_model` re-runs the normal `DecayModel`
constructor path, so the reloaded model is a fresh object with its own lazily-built
JAX kernels -- comparing `.intensity(...)` on the same sample is the meaningful check,
not object identity.

In [3]:
export_path = "tutorial_28_model.json"
export_model(model, export_path)

restored = import_model(export_path)
print("Restored components:", [c.name for c in restored.components])

truth = {p.name: p.value for p in model.parameters}
data = generate_toy(
    model, 800, parameters=truth, seed=28,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)

original_intensity = np.asarray(model.intensity(data.as_dict(), truth))
restored_intensity = np.asarray(restored.intensity(data.as_dict(), truth))
np.testing.assert_allclose(original_intensity, restored_intensity, rtol=1e-12)
print("Reloaded model's intensity matches the original on", data.size, "events.")

Restored components: ['rho', 'f2', 'NR']


Reloaded model's intensity matches the original on 800 events.


## 3. The one unsupported case

`normalization_method="toy-mc"` depends on an external `normalization_sample`
(arbitrary JAX/numpy arrays from an external generator) that is not part of the
specification -- `export_model` raises `ValueError` for that case. Reconstruct that
particular model directly with `DecayModel(..., normalization_sample=...)` in your own
script instead of round-tripping it through JSON.

## Continue learning

See [docs/model_io.md](../../docs/model_io.md) for shared-parameter preservation,
exporting straight from a `FitSession`, and exporting a model right after a fit
(next: [tutorial 30](tutorial_30_model_with_fitted_values.ipynb)).

Return to [the course guide](TUTORIALS.md).